In [2]:
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, welch
from scipy.ndimage import uniform_filter1d
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import Counter
from scipy.stats import linregress, spearmanr
import re
from scipy.stats import shapiro
import scipy.stats as stats
from numpy.polynomial.polynomial import Polynomial

In [3]:
input_folder = "All Data"

In [4]:
# --- Filter functions ---
nperseg = 1024 
noverlap = nperseg // 2

def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def butter_bandstop(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='bandstop')
    return b, a


def apply_filters(signal, fs):
    # Ensure numeric array (coerce bad strings to NaN, then interpolate/zero-fill)
    signal = pd.to_numeric(np.asarray(signal).ravel(), errors='coerce').astype(np.float64)
    if not np.isfinite(signal).all():
        signal = pd.Series(signal).interpolate(limit_direction='both').fillna(0.0).to_numpy()


    # Bandpass 1-120 Hz
    b_bp, a_bp = butter_bandpass(1.0, 120.0, fs, 4)
    filtered = filtfilt(b_bp, a_bp, signal)
    # Notch 40-55 Hz
    b_notch, a_notch = butter_bandstop(40.0, 55.0, fs, 1)
    filtered = filtfilt(b_notch, a_notch, filtered)
    # Smoothing (moving average, 5 ms window)
    window_size = max(1, int(fs * (5 / 1000.0)))
    filtered = uniform_filter1d(filtered, size=window_size)
    return filtered


In [5]:
files = [f for f in os.listdir(input_folder) if f.lower().endswith(".csv")] 

def normalize_trial_name(trial_label):
    """Rename tone_in_noise (and variations) to TIN_old"""
    trial = str(trial_label).strip()
    # Replace tone_in_noise or variations with TIN_old
    trial = re.sub(r'\b(?:tone[_\s-]*in[_\s-]*noise|tonein noise)\b', 'TIN_old', trial, flags=re.IGNORECASE)
    return trial

def extract_db_group(trial_label):
    """Extract dB group from trial name, or None if not found"""
    match = re.search(r'(?:^|[_\s-])(60|80)\s*dB\b', str(trial_label), flags=re.IGNORECASE)
    if match:
        return f"{int(match.group(1))}dB"
    return None

def append_db_suffix(trial_label, db_group):
    """Append dB suffix to trial label if not already present"""
    if not db_group:
        return str(trial_label)
    label = str(trial_label).strip()
    if re.search(r'_(?:60|80)dB$', label, flags=re.IGNORECASE):
        return re.sub(r'_(?:60|80)dB$', f'_{db_group}', label, flags=re.IGNORECASE)
    return f"{label}_{db_group}"

# Dictionaries for different trial types
ppi_values = {}  # For capitalized OFFSET trials (dual recordings)
max_values = {}  # For non-offset/control trials
offset_ppi_trials = {}  # For offset_PPI_X and TIN_X trials (separate recordings)
asr_control_trials = {}  # For ASR control trials split by dB group
repetition_tracker = {}

for filename in files:
    filepath = os.path.join(input_folder, filename)
    
    # Load CSV and skip nonsensical rows
    df = pd.read_csv(filepath, low_memory=False, skiprows=[0, 2])

    # Extract animal number and date from filename
    animal_number = filename[:-4].split('_')[0]
    date = filename[:-4].split('_')[1]
    animal_date_key = f"{animal_number}_{date}"

    # Convert value column to numeric
    df.iloc[:, 19] = pd.to_numeric(df.iloc[:, 19], errors='coerce')
    signal = df.iloc[:, 19].values
    
    # Apply filters
    filtered_signal = apply_filters(signal, 500.0)

    # Take absolute value after filtering 
    filtered_signal = np.abs(filtered_signal)
    df.iloc[:, 19] = filtered_signal
    
    # Get Trial column (column 17, 1-indexed = index 16)
    df['trial'] = df.iloc[:, 16].astype(str).str.strip()
    
    # Normalize trial names (rename tone_in_noise to TIN_old)
    df['trial'] = df['trial'].apply(normalize_trial_name)
    
    # Filter for rows containing 'control', 'offset', or 'tin' (case-insensitive)
    df['is_target'] = df['trial'].str.lower().str.contains('control|offset|tin', na=False)
    df = df[df['is_target']].copy()
    
    if len(df) == 0:
        continue
    
    # Identify segments (consecutive rows with same trial name)
    df['trial_shift'] = df['trial'].shift(1)
    df['new_segment'] = df['trial'] != df['trial_shift']
    df['segment_id'] = df['new_segment'].cumsum()
    
    # Process each segment
    for segment_id, segment_df in df.groupby('segment_id'):
        trial_name = segment_df.iloc[0]['trial']
        trial_name_lower = str(trial_name).lower()
        db_group = extract_db_group(trial_name)
        
        # If no dB suffix found, assume it's 60dB
        if db_group is None:
            db_group = '60dB'
        
        # Create unique key for this animal+trial combination
        normalized_trial_name = append_db_suffix(trial_name, db_group)
        animal_trial_key = f"{animal_number}_{normalized_trial_name}"
        
        # Increment repetition counter
        if animal_trial_key not in repetition_tracker:
            repetition_tracker[animal_trial_key] = 0
        repetition_tracker[animal_trial_key] += 1
        
        unique_key = f"{animal_number}_{normalized_trial_name}_Rep{repetition_tracker[animal_trial_key]}"
        
        # Get max value for this segment
        max_val = segment_df.iloc[:, 19].max()
        
        is_offset_ppi = re.search(r'offset[_\s-]*ppi(?:[_\s-]*(?:\d+|old))?(?=$|[_\s-])', trial_name_lower, flags=re.IGNORECASE) is not None
        is_tin = re.search(r'(?:^|[_\s-])tin(?:[_\s-]*(?:\d+|old))?(?=$|[_\s-])', trial_name_lower, flags=re.IGNORECASE) is not None

        # Check if this is a capitalized OFFSET trial (dual recording with split)
        if 'offset' in trial_name_lower and not is_offset_ppi and not is_tin:
            # Split at MS = 15000 (column 18, 0-indexed)
            segment_df['MS'] = pd.to_numeric(segment_df.iloc[:, 18], errors='coerce')
            split_idx = segment_df[segment_df['MS'] >= 15000].index
            
            if len(split_idx) > 0:
                # First segment: prepulse+startle (PPS)
                first_half = segment_df.loc[:split_idx[0]-1]
                # Second segment: startle alone (S)
                second_half = segment_df.loc[split_idx[0]:]
                
                if len(first_half) > 0 and len(second_half) > 0:
                    PPS = first_half.iloc[:, 19].max()
                    S = second_half.iloc[:, 19].max()
                    
                    # Calculate %PPI = 100 * ((S - PPS) / S)
                    if S != 0:
                        ppi_percent = 100 * ((S - PPS) / S)
                        ppi_values[unique_key] = ppi_percent
                    else:
                        ppi_values[unique_key] = np.nan
                else:
                    ppi_values[unique_key] = max_val
            else:
                ppi_values[unique_key] = max_val
        
        # Check if this is offset_PPI_X trial (separate recording)
        elif is_offset_ppi:
            trial_type_match = re.search(r'offset[_\s-]*ppi[_\s-]*(\d+)(?=$|[_\s-])', trial_name, flags=re.IGNORECASE)
            if trial_type_match:
                suffix = trial_type_match.group(1)
                trial_type = f"offset_PPI_{int(suffix)}"
            else:
                trial_type = 'offset_PPI'

            trial_type = append_db_suffix(trial_type, db_group)
            key = f"{animal_date_key}_{trial_type}"
            if key not in offset_ppi_trials:
                offset_ppi_trials[key] = []
            offset_ppi_trials[key].append(max_val)
        
        # Check if this is a TIN trial (TIN_X variants, including TIN_old)
        elif is_tin:
            tin_match = re.search(r'tin[_\s-]*(\d+|old)(?=$|[_\s-])', trial_name, flags=re.IGNORECASE)
            if tin_match:
                tin_suffix = tin_match.group(1)
                if str(tin_suffix).lower() == 'old':
                    trial_type = 'TIN_old'
                else:
                    trial_type = f"TIN_{int(tin_suffix)}"
            else:
                trial_type = 'TIN'

            trial_type = append_db_suffix(trial_type, db_group)
            key = f"{animal_date_key}_{trial_type}"
            if key not in offset_ppi_trials:
                offset_ppi_trials[key] = []
            offset_ppi_trials[key].append(max_val)
        
        # Check if this is an ASR control trial
        elif 'control' in trial_name_lower and 'asr' in trial_name_lower:
            control_group = db_group if db_group else 'unknown'
            key = f"{animal_date_key}_{control_group}"
            if key not in asr_control_trials:
                asr_control_trials[key] = []
            asr_control_trials[key].append(max_val)
        
        else:
            # Other trials - just save the maximum
            max_values[unique_key] = max_val

# Calculate %PPI for offset_PPI and TIN trials by comparing to matching ASR controls
matched_count = 0
unmatched_count = 0
unmatched_trials = []

print("\n" + "="*70)
print("MATCHING offset_PPI/TIN TRIALS WITH ASR CONTROLS (same animal, date, and dB group)")
print("="*70)

for offset_key, offset_values in offset_ppi_trials.items():
    # Extract animal_date and trial type
    parts = offset_key.split('_')
    animal = parts[0]
    date = parts[1]
    trial_type = '_'.join(parts[2:])  # e.g., offset_PPI_10_60dB or TIN_10_80dB
    animal_date_key = f"{animal}_{date}"
    db_group = extract_db_group(trial_type)
    control_group_key = f"{animal_date_key}_{db_group}" if db_group else f"{animal_date_key}_unknown"
    
    # Get corresponding ASR control values for the same dB group
    if control_group_key in asr_control_trials:
        avg_offset = np.mean(offset_values)
        avg_control = np.mean(asr_control_trials[control_group_key])
        
        # Calculate %PPI = 100 * ((S_control - PPS_offset) / S_control)
        if avg_control != 0:
            ppi_percent = 100 * ((avg_control - avg_offset) / avg_control)
            ppi_key = f"{animal}_{date}_{trial_type}"
            ppi_values[ppi_key] = ppi_percent
            matched_count += 1
            #print(f"✓ {animal} {date} {trial_type}: matched with {len(asr_control_trials[control_group_key])} ASR control(s)")
        else:
            print(f"⚠ {animal} {date} {trial_type}: ASR control = 0, skipping")
            unmatched_count += 1
            unmatched_trials.append(offset_key)
    else:
        #print(f"✗ {animal} {date} {trial_type}: NO MATCHING ASR CONTROL FOUND FOR {db_group if db_group else 'unknown'}")
        unmatched_count += 1
        unmatched_trials.append(offset_key)

print("\n" + "="*70)
print(f"MATCHING SUMMARY:")
print(f"  Successfully matched: {matched_count}")
print(f"  Unmatched/skipped:    {unmatched_count}")
print("="*70)

if unmatched_trials:
    print("\nWARNING: The following offset_PPI/TIN trials had no matching ASR controls:")
    for trial in unmatched_trials:
        print(f"  - {trial}")

print(f"\nProcessed {len(ppi_values)} %PPI values ({sum(1 for k in ppi_values.keys() if ('offset_PPI' in k or 'TIN_' in k))} from offset_PPI/TIN trials).")
print(f"Processed {len(max_values)} other trial segments.")


MATCHING offset_PPI/TIN TRIALS WITH ASR CONTROLS (same animal, date, and dB group)

MATCHING SUMMARY:
  Successfully matched: 400
  Unmatched/skipped:    15

  - Animal56_March3_TIN_4_60dB
  - Animal56_March3_offset_PPI_14_60dB
  - Animal56_March3_TIN_8_60dB
  - Animal56_March3_TIN_old_60dB
  - Animal56_March3_TIN_12_60dB
  - Animal56_March3_offset_PPI_4_60dB
  - Animal56_March3_offset_PPI_8_60dB
  - Animal56_March3_TIN_14_60dB
  - Animal56_March3_offset_PPI_16_60dB
  - Animal56_March3_TIN_6_60dB
  - Animal56_March3_offset_PPI_12_60dB
  - Animal56_March3_offset_PPI_10_60dB
  - Animal56_March3_offset_PPI_6_60dB
  - Animal56_March3_TIN_10_60dB
  - Animal56_March3_TIN_16_60dB

Processed 1580 %PPI values (400 from offset_PPI/TIN trials).
Processed 0 other trial segments.


In [6]:
# Convert to DataFrames
ppi_df = pd.DataFrame(list(ppi_values.items()), columns=['Key', '%PPI'])
max_df = pd.DataFrame(list(max_values.items()), columns=['Key', 'Max_ValueG'])

print("\nSample %PPI entries:")
print(ppi_df.head())
print("\nSample Max Value entries:")
print(max_df.head())

# Calculate mean %PPI per trial
def normalize_db_suffix(text):
    return re.sub(r'_(?:60|80)db$', lambda m: m.group(0).replace('db', 'dB'), str(text), flags=re.IGNORECASE)

def split_db_suffix(trial_label):
    trial_label = str(trial_label)
    m = re.search(r'_(60|80)dB$', trial_label, flags=re.IGNORECASE)
    if not m:
        return trial_label, np.nan
    db_group = f"{int(m.group(1))}dB"
    base = re.sub(r'_(?:60|80)dB$', '', trial_label, flags=re.IGNORECASE)
    return base, db_group

def extract_trial_name(key):
    key = str(key)
    db_match = re.search(r'_(60|80)dB$', key, flags=re.IGNORECASE)
    db_suffix = f"_{int(db_match.group(1))}dB" if db_match else ""
    key_no_db = re.sub(r'_(?:60|80)dB$', '', key, flags=re.IGNORECASE)

    # Handle matched offset_PPI trials (format: Animal_Date_offset_PPI_X[_60dB/_80dB])
    if re.search(r'offset[_\s-]*PPI', key_no_db, flags=re.IGNORECASE):
        match = re.search(r'(offset[_\s-]*PPI[_\s-]*(?:\d+|old))(?:$|_)', key_no_db, flags=re.IGNORECASE)
        if match:
            normalized = re.sub(r'offset[_\s-]*PPI', 'offset_PPI', match.group(1), flags=re.IGNORECASE)
            normalized = re.sub(r'offset_PPI[_\s-]*(\d+|old)$', r'offset_PPI_\1', normalized, flags=re.IGNORECASE)
            return normalize_db_suffix(normalized + db_suffix)
        if db_suffix:
            return f"offset_PPI{db_suffix}"
        return 'offset_PPI'

    # Handle matched TIN trials (format: Animal_Date_TIN_X[_60dB/_80dB] or TIN_old[_60dB/_80dB])
    if re.search(r'(?:^|_)TIN(?:_|$)', key_no_db, flags=re.IGNORECASE):
        match = re.search(r'(TIN_(?:\d+|old))(?:$|_)', key_no_db, flags=re.IGNORECASE)
        if match:
            return normalize_db_suffix(match.group(1) + db_suffix)
        if db_suffix:
            return f"TIN{db_suffix}"
        return 'TIN'

    # Format: Animal_Trial_RepN - extract Trial part
    match = re.search(r'Animal\d+_(.+)_Rep\d+', key, flags=re.IGNORECASE)
    if match:
        return normalize_db_suffix(match.group(1))
    return normalize_db_suffix(key)

ppi_df['Trial'] = ppi_df['Key'].apply(extract_trial_name)
ppi_df[['Trial_Base', 'dB_Group']] = ppi_df['Trial'].apply(lambda t: pd.Series(split_db_suffix(t)))
ppi_df['dB_Group'] = ppi_df['dB_Group'].astype('object')

# Numeric offset length is extracted only from trial base, never from dB suffix
ppi_df['Offset_Length'] = ppi_df['Trial_Base'].str.extract(
    r'(?:^|_)(?:OFFSET|offset[_\s-]*PPI|TIN)[_\s-]*(\d+)(?:$|_)', flags=re.IGNORECASE
)[0].astype(float)

# Remove negative %PPI values
pre_neg_count = len(ppi_df)
ppi_df = ppi_df[ppi_df['%PPI'] >= 0].reset_index(drop=True)
removed_neg = pre_neg_count - len(ppi_df)
print(f"\nRemoved {removed_neg} negative %PPI values.")

# Remove outliers within each trial (IQR-based)
def filter_outliers_iqr(group, value_col='%PPI', k=1.5, min_n=4):
    if len(group) < min_n:
        return group
    q1 = group[value_col].quantile(0.25)
    q3 = group[value_col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0 or not np.isfinite(iqr):
        return group
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return group[group[value_col].between(lower, upper)]

pre_outlier_count = len(ppi_df)
ppi_df = ppi_df.groupby('Trial', group_keys=False).apply(filter_outliers_iqr).reset_index(drop=True)
removed = pre_outlier_count - len(ppi_df)
print(f"\nRemoved {removed} %PPI outliers across trials (IQR-based).")

# Group by trial and calculate mean
mean_ppi_per_trial = ppi_df.groupby('Trial')['%PPI'].mean().sort_values(ascending=False)

print("\n\nMean %PPI per Trial:")
print("=" * 50)
for trial, mean_ppi in mean_ppi_per_trial.items():
    print(f"{trial:30s}: {mean_ppi:7.2f}%")


Sample %PPI entries:
                             Key       %PPI
0  Animal10_offset_ASR_60dB_Rep1 -29.895575
1  Animal10_offset_ASR_60dB_Rep2  -8.271550
2  Animal10_offset_ASR_60dB_Rep3  -0.188765
3  Animal10_offset_ASR_60dB_Rep4  22.520019
4  Animal10_offset_ASR_60dB_Rep5   8.679570

Sample Max Value entries:
Empty DataFrame
Columns: [Key, Max_ValueG]
Index: []

Removed 350 negative %PPI values.

Removed 11 %PPI outliers across trials (IQR-based).


Mean %PPI per Trial:
offset_PPI_10_80dB            :   60.11%
offset_PPI_16_80dB            :   56.54%
OFFSET_16_80dB                :   53.62%
TIN_4_60dB                    :   53.14%
TIN_10_60dB                   :   48.93%
OFFSET_14_60dB                :   43.60%
OFFSET_10_80dB                :   42.69%
TIN_16_60dB                   :   42.52%
OFFSET_10_60dB                :   41.97%
OFFSET_16_60dB                :   41.57%
OFFSET_12_60dB                :   39.43%
OFFSET_8_60dB                 :   39.39%
TIN_12_60dB                   :

C:\Users\Galahad\AppData\Local\Temp\ipykernel_20996\3830527477.py:84: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ppi_df = ppi_df.groupby('Trial', group_keys=False).apply(filter_outliers_iqr).reset_index(drop=True)


In [12]:
# Create Excel file with %PPI data organized by animal and trial
# Extract animal number and date from Key column
ppi_df['Animal'] = ppi_df['Key'].str.extract(r'(Animal\d+)')[0]
ppi_df['Date'] = ppi_df['Key'].str.extract(r'Animal\d+_(\w+)')[0]

# Create a working dataframe with the necessary columns
export_df = ppi_df[['Animal', 'Date', 'dB_Group', 'Trial', '%PPI']].copy()

# Remove the dB suffix from trial names for grouping
# This converts "offset_PPI_10_60dB" to "offset_PPI_10"
export_df['Trial_Base'] = export_df['Trial'].str.replace(r'_(?:60|80)dB$', '', regex=True, flags=re.IGNORECASE)

# For each animal-trial combination, we might have multiple %PPI values
# Let's take the mean %PPI per Animal-Date-Trial-dB combination
grouped_data = export_df.groupby(['Animal', 'Date', 'dB_Group', 'Trial_Base'])['%PPI'].mean().reset_index()

# Separate 60dB and 80dB data
data_60db = grouped_data[grouped_data['dB_Group'] == '60dB'].copy()
data_80db = grouped_data[grouped_data['dB_Group'] == '80dB'].copy()

# Pivot both separately (using Trial_Base as columns)
pivot_60 = data_60db.pivot_table(
    index=['Animal', 'Date'], 
    columns='Trial_Base', 
    values='%PPI',
    aggfunc='mean'
)

pivot_80 = data_80db.pivot_table(
    index=['Animal', 'Date'], 
    columns='Trial_Base', 
    values='%PPI',
    aggfunc='mean'
)

# Get all unique trials (union of both)
all_trials = sorted(set(list(pivot_60.columns) + list(pivot_80.columns)))

# Sort trials according to specified order:
# TIN_old, then TIN trials, then offset_PPI trials, then OFFSET trials
def sort_trials(trial_name):
    """
    Returns a tuple for sorting:
    (category_order, numeric_value, trial_name)
    """
    trial = str(trial_name)
    
    # TIN_old comes first
    if trial == 'TIN_old':
        return (0, 0, trial)
    
    # TIN trials (sorted by number)
    if trial.startswith('TIN_'):
        match = re.search(r'TIN_(\d+)', trial)
        if match:
            return (1, int(match.group(1)), trial)
        return (1, 999999, trial)
    
    # offset_PPI trials (sorted by number)
    if 'offset_PPI' in trial:
        match = re.search(r'offset_PPI_(\d+)', trial, re.IGNORECASE)
        if match:
            return (2, int(match.group(1)), trial)
        if 'old' in trial.lower():
            return (2, 0, trial)
        return (2, 999999, trial)
    
    # OFFSET trials (sorted by number)
    if 'OFFSET' in trial:
        match = re.search(r'OFFSET[_\s-]*(\d+)', trial, re.IGNORECASE)
        if match:
            return (3, int(match.group(1)), trial)
        return (3, 999999, trial)
    
    # Everything else
    return (4, 999999, trial)

all_trials_sorted = sorted(all_trials, key=sort_trials)

# Determine which animals have both 60dB and 80dB data
animals_with_60db = set(grouped_data[grouped_data['dB_Group'] == '60dB']['Animal'])
animals_with_80db = set(grouped_data[grouped_data['dB_Group'] == '80dB']['Animal'])
animals_with_both = animals_with_60db & animals_with_80db

print(f"\nAnimals with only 60dB: {sorted(animals_with_60db - animals_with_both, key=lambda x: int(x.replace('Animal', '')))}")
print(f"Animals with only 80dB: {sorted(animals_with_80db - animals_with_both, key=lambda x: int(x.replace('Animal', '')))}")
print(f"Animals with both dB levels: {sorted(animals_with_both, key=lambda x: int(x.replace('Animal', '')))}")

# Create the final dataframe
result_rows = []

for animal in sorted(grouped_data['Animal'].unique(), key=lambda x: int(x.replace('Animal', ''))):
    # Get all dates for this animal
    animal_data = grouped_data[grouped_data['Animal'] == animal]
    
    # Determine if this animal has both dB levels
    has_both = animal in animals_with_both
    
    # For simplicity, we'll combine all data for each animal into one row
    row = {'Animal': animal, 'Sex': ''}
    
    for trial in all_trials_sorted:
        val_60 = ''
        val_80 = ''
        
        # Get 60dB value for this animal-trial
        if (animal, trial) in [(idx[0], col) for idx in pivot_60.index for col in pivot_60.columns]:
            animal_60_data = pivot_60.loc[pivot_60.index.get_level_values('Animal') == animal]
            if trial in animal_60_data.columns and not animal_60_data[trial].isna().all():
                val_60 = f"{animal_60_data[trial].mean():.2f}"
        
        # Get 80dB value for this animal-trial
        if (animal, trial) in [(idx[0], col) for idx in pivot_80.index for col in pivot_80.columns]:
            animal_80_data = pivot_80.loc[pivot_80.index.get_level_values('Animal') == animal]
            if trial in animal_80_data.columns and not animal_80_data[trial].isna().all():
                val_80 = f"{animal_80_data[trial].mean():.2f}"
        
        # Format cell value based on whether animal has both dB levels
        if has_both:
            # Use "/" format for animals with both dB levels
            if val_60 or val_80:
                display_60 = val_60 if val_60 else 'n.a.'
                display_80 = val_80 if val_80 else 'n.a.'
                row[trial] = f"{display_60} / {display_80}"
            else:
                row[trial] = ''
        else:
            # Single value for animals with only one dB level
            if val_60:
                row[trial] = val_60
            elif val_80:
                row[trial] = val_80
            else:
                row[trial] = ''
    
    result_rows.append(row)

# Create final dataframe
final_df = pd.DataFrame(result_rows)

# Reorder columns: Animal, Sex, then trials (in sorted order)
cols = ['Animal', 'Sex'] + [col for col in all_trials_sorted if col in final_df.columns]
final_df = final_df[cols]

# Define output file path
output_file = os.path.join('Output', 'PPI_Analysis_by_Animal.xlsx')
os.makedirs('Output', exist_ok=True)

# Export to Excel with proper formatting to prevent date interpretation
print(f"\nCreating Excel file (will replace if exists)...")
with pd.ExcelWriter(output_file, engine='xlsxwriter', mode='w') as writer:
    final_df.to_excel(writer, index=False, sheet_name='PPI Data')
    
    # Get the worksheet and workbook objects
    workbook = writer.book
    worksheet = writer.sheets['PPI Data']
    
    # Define a text format to prevent Excel from interpreting values as dates
    text_format = workbook.add_format({'num_format': '@'})
    
    # Apply text format to all columns
    for col_num, col_name in enumerate(final_df.columns):
        worksheet.set_column(col_num, col_num, 15, text_format)

print(f"\n{'='*70}")
print(f"Excel file saved successfully!")
print(f"File saved to: {output_file}")
print(f"{'='*70}")
print(f"\nDataset summary:")
print(f"  Total animals: {final_df['Animal'].nunique()}")
print(f"  Total trial columns: {len(final_df.columns) - 2}")  # Subtract Animal, Sex columns
print(f"  Rows in output: {len(final_df)}")
print(f"\nColumn order: TIN_old, TIN trials (by number), offset_PPI trials (by number), OFFSET trials (by number)")
print(f"\nNote: Animals with BOTH 60dB and 80dB data show '60dB value / 80dB value'")
print(f"Note: Animals with ONLY one dB level show single values (no '/')")
print(f"Note: Missing values in '/' format are shown as 'n.a.'")
print(f"Note: The 'Sex' column is empty. Please fill it in manually in Excel.")
print(f"Note: Re-running this cell will REPLACE the existing file (no merging).")
print(f"\nFirst few rows of the output:")
print(final_df.head())


Animals with only 60dB: ['Animal1', 'Animal2', 'Animal4', 'Animal6', 'Animal10', 'Animal15', 'Animal25', 'Animal26', 'Animal27', 'Animal28', 'Animal43', 'Animal44', 'Animal45', 'Animal46', 'Animal47', 'Animal48', 'Animal49', 'Animal56', 'Animal93']
Animals with only 80dB: []
Animals with both dB levels: ['Animal57', 'Animal58', 'Animal59', 'Animal60', 'Animal62', 'Animal65']

Creating Excel file (will replace if exists)...

Excel file saved successfully!
File saved to: Output\PPI_Analysis_by_Animal.xlsx

Dataset summary:
  Total animals: 25
  Total trial columns: 26
  Rows in output: 25

Column order: TIN_old, TIN trials (by number), offset_PPI trials (by number), OFFSET trials (by number)

Note: Animals with BOTH 60dB and 80dB data show '60dB value / 80dB value'
Note: Animals with ONLY one dB level show single values (no '/')
Note: Missing values in '/' format are shown as 'n.a.'
Note: The 'Sex' column is empty. Please fill it in manually in Excel.
Note: Re-running this cell will REP

In [13]:
# ============================================================================
# VERIFICATION: Check specific entries to verify data processing
# ============================================================================

def verify_entry(animal, trial, show_details=True):
    """
    Verify a specific animal-trial combination by showing:
    1. Raw data values from offset_ppi_trials or ppi_values
    2. Processed %PPI value
    3. Final Excel output value
    """
    print(f"\n{'='*70}")
    print(f"VERIFICATION: {animal} - {trial}")
    print(f"{'='*70}")
    
    # Check in offset_ppi_trials (for offset_PPI and TIN trials)
    found = False
    for key, values in offset_ppi_trials.items():
        if animal in key and trial in key:
            print(f"\n1. Raw offset_ppi_trials entry:")
            print(f"   Key: {key}")
            print(f"   Values: {values}")
            print(f"   Mean: {np.mean(values):.4f}")
            found = True
            
            # Find matching ASR control
            parts = key.split('_')
            animal_num = parts[0]
            date = parts[1]
            trial_type = '_'.join(parts[2:])
            animal_date_key = f"{animal_num}_{date}"
            db_group = extract_db_group(trial_type)
            control_key = f"{animal_date_key}_{db_group}" if db_group else f"{animal_date_key}_unknown"
            
            if control_key in asr_control_trials:
                print(f"\n2. Matching ASR control:")
                print(f"   Key: {control_key}")
                print(f"   Values: {asr_control_trials[control_key]}")
                print(f"   Mean: {np.mean(asr_control_trials[control_key]):.4f}")
                
                # Calculate %PPI
                avg_offset = np.mean(values)
                avg_control = np.mean(asr_control_trials[control_key])
                if avg_control != 0:
                    ppi_percent = 100 * ((avg_control - avg_offset) / avg_control)
                    print(f"\n3. Calculated %PPI:")
                    print(f"   Formula: 100 * ((ASR_control - offset) / ASR_control)")
                    print(f"   = 100 * (({avg_control:.4f} - {avg_offset:.4f}) / {avg_control:.4f})")
                    print(f"   = {ppi_percent:.2f}%")
            else:
                print(f"\n2. NO MATCHING ASR CONTROL FOUND for {db_group if db_group else 'unknown'}")
    
    # Check in ppi_values (processed %PPI)
    if show_details:
        print(f"\n4. Processed ppi_values entries:")
        matching_keys = [k for k in ppi_values.keys() if animal in k and trial in k]
        if matching_keys:
            for key in matching_keys:
                print(f"   {key}: {ppi_values[key]:.2f}%")
        else:
            print(f"   No matching entries found")
    
    # Check in ppi_df (after outlier removal)
    print(f"\n5. In ppi_df (after outlier filtering):")
    animal_trial_df = ppi_df[(ppi_df['Animal'] == animal) & (ppi_df['Trial'].str.contains(trial, case=False, na=False))]
    if len(animal_trial_df) > 0:
        print(animal_trial_df[['Animal', 'Date', 'Trial', 'dB_Group', '%PPI']].to_string(index=False))
    else:
        print(f"   No entries found in ppi_df")
    
    # Check in final Excel output
    print(f"\n6. In final Excel output:")
    animal_row = final_df[final_df['Animal'] == animal]
    if len(animal_row) > 0:
        # Find columns matching the trial
        matching_cols = [col for col in final_df.columns if trial.replace('_', '') in col.replace('_', '')]
        if matching_cols:
            print(f"   Animal: {animal}")
            for col in matching_cols:
                val = animal_row.iloc[0][col]
                print(f"   {col}: {val}")
        else:
            print(f"   Trial column not found in Excel output")
    else:
        print(f"   Animal not found in Excel output")
    
    print(f"{'='*70}\n")
    return found

# Examples - you can modify these to check different entries
verify_entry('Animal93', 'offset_PPI_8')

# You can also list all available trials for a specific animal
print("\n" + "="*70)
print("ALL TRIALS FOR SPECIFIC ANIMAL")
print("="*70)
animal_to_check = 'Animal93'
animal_trials = ppi_df[ppi_df['Animal'] == animal_to_check]['Trial'].unique()
print(f"\n{animal_to_check} has {len(animal_trials)} unique trials:")
for trial in sorted(animal_trials):
    print(f"  - {trial}")


VERIFICATION: Animal93 - offset_PPI_8

1. Raw offset_ppi_trials entry:
   Key: Animal93_January21_offset_PPI_8_60dB
   Values: [np.float64(51.61980048330308), np.float64(59.68685948751913), np.float64(58.72776226810569), np.float64(55.521575616837936), np.float64(22.309850404803186), np.float64(12.227809107373595), np.float64(19.900802136904467), np.float64(61.62182698471369), np.float64(27.645492988777356), np.float64(49.754953952376795)]
   Mean: 41.9017

2. Matching ASR control:
   Key: Animal93_January21_60dB
   Values: [np.float64(59.62389123907931), np.float64(49.1682688964419), np.float64(65.12571962272428), np.float64(62.84086720121513), np.float64(24.705956907735832), np.float64(22.438277029905095), np.float64(36.51472170726388), np.float64(20.579359999819673), np.float64(55.66047515536361), np.float64(50.55162432530648)]
   Mean: 44.7209

3. Calculated %PPI:
   Formula: 100 * ((ASR_control - offset) / ASR_control)
   = 100 * ((44.7209 - 41.9017) / 44.7209)
   = 6.30%

4. Pro

### Match Animal Number with Animal ID

In [41]:
import os
import re
import pandas as pd

results = []
for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith('.csv'):
        continue

    filepath = os.path.join(input_folder, filename)
    first_num_match = re.search(r'\d+', filename)
    file_number = int(first_num_match.group()) if first_num_match else filename

    try:
        # Skip the first row when identifying headers.
        header_df = pd.read_csv(filepath, nrows=0, skiprows=[0])
        normalized_cols = {str(c).strip().lower(): c for c in header_df.columns}
        group_col = normalized_cols.get('group')

        if group_col is None:
            group_value = 'Group column not found'
        else:
            # Read only Group column, also skipping first row.
            df_group = pd.read_csv(
                filepath,
                skiprows=[0],
                usecols=[group_col],
                dtype={group_col: 'string'},
                low_memory=False
            )
            group_value = df_group[group_col].iloc[9]

        results.append({'Animal Number': file_number, 'ID': group_value})
    except Exception as e:
        results.append({'Animal Number': file_number, 'ID': f'Error: {e}'})

group_row10_df = pd.DataFrame(results)
group_row10_df = group_row10_df.drop_duplicates(subset=['Animal Number', 'ID']).reset_index(drop=True)
display(group_row10_df)

,Animal Number,ID
0,10,10
1,15,15
2,1,1
3,25,<NA>
4,26,<NA>
5,27,<NA>
6,28,<NA>
7,2,2
8,43,018243
9,44,018244
